In [ ]:
# augmentation.py
import os
import json
from collections import Counter
import random
from tqdm import tqdm
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from concurrent.futures import ThreadPoolExecutor, as_completed

# --- 설정 (Configuration) ---
VQA_DATA_DIR = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/dataset/VQA_v2"
OUTPUT_DIR = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/dataset/VQA_v2/generated_vqa_data_abcd"
MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct" # **대회 규칙 재확인 필수! (2023.12.31 이전 공개, 3B 미만)**
NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 1
print(f"Detected {NUM_GPUS} GPUs.")

MAX_TRAIN_SAMPLES = 10000 # 훈련 데이터 제한
MAX_VAL_SAMPLES = 1000   # 검증 데이터 제한

MAX_NEW_TOKENS = 50
TEMPERATURE = 0.7
CHOICE_LABELS = ["A", "B", "C", "D"]

# --- 디렉토리 생성 ---
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- VQA 데이터 로드 함수 ---
def load_vqa_data(split="train"):
    if split == "train":
        questions_path = os.path.join(VQA_DATA_DIR, "annotation", "v2_OpenEnded_mscoco_train2014_questions.json")
        annotations_path = os.path.join(VQA_DATA_DIR, "annotation", "v2_mscoco_train2014_annotations.json")
    elif split == "val":
        questions_path = os.path.join(VQA_DATA_DIR, "annotation", "v2_OpenEnded_mscoco_val2014_questions.json")
        annotations_path = os.path.join(VQA_DATA_DIR, "annotation", "v2_mscoco_val2014_annotations.json")
    else:
        raise ValueError("Split must be 'train' or 'val'")

    print(f"Loading questions from: {questions_path}")
    print(f"Loading annotations from: {annotations_path}")

    with open(questions_path, 'r', encoding='utf-8') as f:
        questions = json.load(f)['questions']
    with open(annotations_path, 'r', encoding='utf-8') as f:
        annotations = json.load(f)['annotations']

    qa_data = {}
    for q in questions:
        qa_data[q['question_id']] = {
            'image_id': q['image_id'],
            'question': q['question'],
            'raw_answers': []
        }

    for ann_item in annotations:
        question_id = ann_item['question_id']
        if question_id in qa_data:
            for answer_obj in ann_item['answers']:
                qa_data[question_id]['raw_answers'].append(answer_obj['answer'])

    processed_qa_data = []
    for q_id, data in qa_data.items():
        answers = [ans.lower() for ans in data['raw_answers'] if ans.strip()]
        if not answers:
            continue

        most_common_answer, count = Counter(answers).most_common(1)[0]
        
        processed_qa_data.append({
            'question_id': q_id,
            'image_id': data['image_id'],
            'question': data['question'],
            'correct_answer': most_common_answer
        })
    print(f"Loaded and processed {len(processed_qa_data)} questions for {split} split.")
    return processed_qa_data


# --- LLM 및 파이프라인 로드 (각 GPU별로) ---
global_tokenizers = []
global_generators = []
global_models = [] 

print(f"Loading {NUM_GPUS} LLM instances for parallel processing...")
try:
    for i in range(NUM_GPUS):
        print(f"Loading LLM instance {i+1}/{NUM_GPUS} for cuda:{i}...")
        
        tokenizer_instance = AutoTokenizer.from_pretrained(MODEL_ID)
        if tokenizer_instance.pad_token is None:
            tokenizer_instance.pad_token = tokenizer_instance.eos_token
        if tokenizer_instance.model_max_length > 1e5:
            tokenizer_instance.model_max_length = 2048
        global_tokenizers.append(tokenizer_instance)

        model_llm_instance = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            torch_dtype=torch.bfloat16 
        ).to(f"cuda:{i}") # 명시적으로 각 GPU에 모델 로드
        model_llm_instance.eval()
        global_models.append(model_llm_instance) 

        generator_instance = pipeline(
            "text-generation",
            model=model_llm_instance,
            tokenizer=tokenizer_instance,
        )
        global_generators.append(generator_instance)
    print("All LLM instances loaded successfully.")
except Exception as e:
    print(f"Error loading LLM: {e}")
    print("Please ensure you have authenticated with Hugging Face Hub (`huggingface-cli login`) if it's a gated model.")
    print("Also, check your GPU memory (each GPU needs enough for the model).")
    exit()

# --- 프롬프트 생성 함수 ---
def create_prompt(question, correct_answer, tokenizer_instance):
    system_message = (
        "You are an AI assistant tasked with generating plausible but incorrect answer choices "
        "for a given question about an image. The correct answer is also provided. "
        "Your goal is to generate 3 distinct incorrect answers."
    )
    user_message = (
        f"Question: \"{question}\"\n"
        f"Correct Answer: \"{correct_answer}\"\n\n"
        "Generate 3 incorrect answers that are:\n"
        "1. Plausible: They should seem like they *could* be correct at first glance.\n"
        "2. Incorrect: They must not be the true correct answer.\n"
        "3. Distinct: They should be different from each other.\n"
        "4. Relevant: They should be related to the question.\n\n"
        "Provide only the 3 incorrect answers, each on a new line, prefixed with a number (e.g., '1. Answer')."
    )

    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
    ]
    prompt = tokenizer_instance.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    return prompt

# --- 오답 생성 및 파싱 함수 ---
def generate_and_parse_false_answers(prompts, generator_pipeline, tokenizer_instance):
    generated_texts = generator_pipeline(
        prompts,
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=TEMPERATURE,
        do_sample=True,
        pad_token_id=tokenizer_instance.pad_token_id,
        return_full_text=False
    )

    parsed_false_answers_batch = []
    for i, generated_text_list in enumerate(generated_texts):
        text = generated_text_list[0]['generated_text'].strip()
        false_answers = []
        lines = text.split('\n')
        for line in lines:
            line = line.strip()
            if line and (line[0].isdigit() and '. ' in line):
                ans = line.split('. ', 1)[1].strip()
            else:
                ans = line.strip()

            if ans.startswith('"') and ans.endswith('"'):
                ans = ans[1:-1]
            
            if ans:
                false_answers.append(ans)
            
            if len(false_answers) >= 3:
                break
        parsed_false_answers_batch.append(false_answers[:3])

    return parsed_false_answers_batch

# --- 개별 데이터 항목 처리 함수 (병렬 처리를 위해) ---
def process_single_item(item_idx, item, generator_pipeline, tokenizer_instance):
    """
    하나의 VQA 질문 항목에 대해 오답을 생성하고 4지선다형으로 구성합니다.
    """
    prompt = create_prompt(item['question'], item['correct_answer'], tokenizer_instance)
    
    false_answers_batch = generate_and_parse_false_answers([prompt], generator_pipeline, tokenizer_instance)
    false_answers = false_answers_batch[0]

    final_false_answers = []
    correct_ans_lower = item['correct_answer'].lower()
    for fa in false_answers:
        if fa.lower() != correct_ans_lower and fa.strip():
            final_false_answers.append(fa)
        if len(final_false_answers) >= 3:
            break
    
    if len(final_false_answers) < 3:
        return None 

    choices_raw = [item['correct_answer']] + final_false_answers
    random.shuffle(choices_raw)

    choices_abcd = [f"{CHOICE_LABELS[k]}. {choice}" for k, choice in enumerate(choices_raw)]
    answer_index = choices_raw.index(item['correct_answer'])

    return {
        'question_id': item['question_id'],
        'image_id': item['image_id'],
        'question': item['question'],
        'choices': choices_abcd,
        'answer_index': answer_index,
        'original_correct_answer': item['correct_answer']
    }


# --- 메인 처리 로직 (병렬 처리 적용) ---
def main():
    for split in ["train", "val"]:
        print(f"\n--- Processing {split} dataset ---")
        vqa_data = load_vqa_data(split=split)
        
        # --- 데이터셋 크기 제한 ---
        if split == "train":
            random.shuffle(vqa_data) 
            vqa_data = vqa_data[:MAX_TRAIN_SAMPLES]
            print(f"Limiting train data to {len(vqa_data)} samples.")
        elif split == "val":
            random.shuffle(vqa_data)
            vqa_data = vqa_data[:MAX_VAL_SAMPLES]
            print(f"Limiting val data to {len(vqa_data)} samples.")

        generated_qa_pairs = []
        
        with ThreadPoolExecutor(max_workers=NUM_GPUS) as executor:
            futures = []
            
            for i, item in enumerate(vqa_data):
                gpu_idx = i % NUM_GPUS
                futures.append(executor.submit(process_single_item, i, item, global_generators[gpu_idx], global_tokenizers[gpu_idx]))

            for future in tqdm(as_completed(futures), total=len(vqa_data), desc=f"Generating {split} 4-choice QAs"):
                result = future.result()
                if result is not None:
                    generated_qa_pairs.append(result)
        
        output_filename = os.path.join(OUTPUT_DIR, f"vqa_v2_{split}_4choice_abcd_generated.json")
        with open(output_filename, 'w', encoding='utf-8') as f:
            json.dump(generated_qa_pairs, f, indent=4, ensure_ascii=False)
        print(f"Generated {len(generated_qa_pairs)} 4-choice QA pairs for {split} split and saved to {output_filename}")

if __name__ == "__main__":
    main()